In [10]:
import sqlite3
from pathlib import Path
import logging
from itertools import combinations

log_path = Path("etl_sdm.txt")

# Dit format zorgt voor de tijdstempels en de mooie weergave!
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | __main__:%(funcName)s:%(lineno)d - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler()
    ],
)

logging.info("SDM-laadproces gestart. Vereiste bibliotheken zijn geïmporteerd.")

2026-04-22 14:01:46 | INFO     | __main__:<module>:19 - SDM-laadproces gestart. Vereiste bibliotheken zijn geïmporteerd.


In [11]:
# BASE verwijst naar de huidige map waarin het script wordt uitgevoerd
BASE = Path.cwd()

# Pad naar de centrale SDM-database
SDM = BASE / 'database/SDM.db'

# Pad naar het tekstbestand met het databaseschema
SCHEMA = BASE / 'ERD/BikeToDrive_RIM - SDM.txt'

# Overzicht van alle bron-databases
# De sleutel is een korte naam, de waarde is het pad naar het .db-bestand
SOURCES = {
    'accessoire_inkoop': BASE / 'database/BikeToDrive_4_Accessoire_Inkoop.db',
    'accessoireverkoop': BASE / 'database/BikeToDrive_1_Accessoireverkoop.db',
    'onderhoud': BASE / 'database/BikeToDrive_3_Onderhoud.db',
    'fiets_inkoop': BASE / 'database/BikeToDrive_5_Fiets_Inkoop.db',
    'fietsverkoop': BASE / 'database/BikeToDrive_2_Fietsverkoop.db',
}

# Opbouw per tuple:
# (bron_database, bron_tabel, doel_tabel_in_SDM)
MAPS = [
    ('accessoire_inkoop', 'Leverancier', 'Accessoire_Inkoop_Leverancier'),
    ('accessoire_inkoop', 'Accessoire', 'Accessoire_Inkoop_Accessoire'),
    ('accessoire_inkoop', 'Accessoire_Inkoop', 'Accessoire_Inkoop'),

    ('accessoireverkoop', 'Filiaal', 'Accessoireverkoop_Filiaal'),
    ('accessoireverkoop', 'Leverancier', 'Accessoireverkoop_Leverancier'),
    ('accessoireverkoop', 'Klant', 'Accessoireverkoop_Klant'),
    ('accessoireverkoop', 'Monteur', 'Accessoireverkoop_Monteur'),
    ('accessoireverkoop', 'Accessoire', 'Accessoireverkoop_Accessoire'),
    ('accessoireverkoop', 'Accessoire_Verkoop', 'Accessoireverkoop_Accessoire_Verkoop'),

    ('onderhoud', 'Fabrikant', 'Onderhoud_Fabrikant'),
    ('onderhoud', 'Filiaal', 'Onderhoud_Filiaal'),
    ('onderhoud', 'Fiets', 'Onderhoud_Fiets'),
    ('onderhoud', 'Monteur', 'Onderhoud_Monteur'),
    ('onderhoud', 'Onderhoud', 'Onderhoud'),

    ('fiets_inkoop', 'Fabrikant', 'Fiets_Inkoop_Fabrikant'),
    ('fiets_inkoop', 'Fiets', 'Fiets_Inkoop_Fiets'),
    ('fiets_inkoop', 'Fiets_Inkoop', 'Fiets_Inkoop'),

    ('fietsverkoop', 'Filiaal', 'Fietsverkoop_Filiaal'),
    ('fietsverkoop', 'Klant', 'Fietsverkoop_Klant'),
    ('fietsverkoop', 'Fabrikant', 'Fietsverkoop_Fabrikant'),
    ('fietsverkoop', 'Monteur', 'Fietsverkoop_Monteur'),
    ('fietsverkoop', 'Fiets', 'Fietsverkoop_Fiets'),
    ('fietsverkoop', 'Fiets_Verkoop', 'Fietsverkoop_Fiets_Verkoop'),
]

In [12]:
# Zet een naam tussen dubbele aanhalingstekens
def q(name):
    return f'"{name}"'

In [13]:
# Verbind met SDM
sdm = sqlite3.connect(SDM)

# Zet foreign keys aan
sdm.execute('PRAGMA foreign_keys = ON')

# Verbind met bron-databases
sources = {k: sqlite3.connect(v) for k, v in SOURCES.items()}

In [14]:
# --- Step 2: bepaal database-overschrijdende associaties ---
# We kijken per tabel die in meerdere bron-databases voorkomt:
# - 1-op-1   : dezelfde sleutelwaarden bestaan in beide databases
# - 1-op-0..1: sleutelwaarden zijn uniek, maar niet overal aanwezig

def get_tables(conn):
    return [
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()
    ]


def get_pk_columns(conn, table_name):
    info = conn.execute(f'PRAGMA table_info({q(table_name)})').fetchall()
    pk_cols = [row[1] for row in sorted(info, key=lambda r: r[5]) if row[5] > 0]
    return pk_cols


def get_key_columns(conn, table_name):
    pk_cols = get_pk_columns(conn, table_name)
    if pk_cols:
        return pk_cols

    # Fallback: gebruik eerste kolom als er geen expliciete primary key is
    info = conn.execute(f'PRAGMA table_info({q(table_name)})').fetchall()
    if info:
        return [info[0][1]]

    return []


def get_key_rows(conn, table_name, key_columns):
    cols_sql = ', '.join(q(col) for col in key_columns)
    rows = conn.execute(
        f'SELECT {cols_sql} FROM {q(table_name)}'
    ).fetchall()
    return rows


def bepaal_relatie(rows_a, rows_b):
    set_a = set(rows_a)
    set_b = set(rows_b)

    if len(rows_a) != len(set_a) or len(rows_b) != len(set_b):
        return 'geen 1-op-1 / 1-op-0..1 (sleutel niet uniek)'

    if set_a == set_b:
        return '1-op-1'

    return '1-op-0..1'


# Verzamel tabellen per database
tables_per_db = {db_name: set(get_tables(conn)) for db_name, conn in sources.items()}

# Welke tabellen komen in meerdere databases voor?
all_tables = sorted(set().union(*tables_per_db.values()))
shared_tables = [
    table_name
    for table_name in all_tables
    if sum(table_name in tables for tables in tables_per_db.values()) > 1
]

if not shared_tables:
    print('Geen gedeelde tabellen gevonden tussen de bron-databases.')
else:
    print('Database-overschrijdende associaties:')
    print('-' * 60)

    for table_name in shared_tables:
        dbs_with_table = [
            db_name for db_name, tables in tables_per_db.items()
            if table_name in tables
        ]

        print(f'\nTabel: {table_name}')

        for db_a, db_b in combinations(dbs_with_table, 2):
            conn_a = sources[db_a]
            conn_b = sources[db_b]

            keys_a = get_key_columns(conn_a, table_name)
            keys_b = get_key_columns(conn_b, table_name)

            if keys_a != keys_b:
                print(f'  {db_a} <-> {db_b}: overgeslagen (sleutels verschillen: {keys_a} vs {keys_b})')
                continue

            if not keys_a:
                print(f'  {db_a} <-> {db_b}: overgeslagen (geen sleutel gevonden)')
                continue

            rows_a = get_key_rows(conn_a, table_name, keys_a)
            rows_b = get_key_rows(conn_b, table_name, keys_b)

            relatie = bepaal_relatie(rows_a, rows_b)
            print(f'  {db_a} <-> {db_b}: {relatie} via {keys_a}')


Database-overschrijdende associaties:
------------------------------------------------------------

Tabel: Accessoire
  accessoire_inkoop <-> accessoireverkoop: 1-op-0..1 via ['accessoirenr']

Tabel: Fabrikant
  onderhoud <-> fiets_inkoop: 1-op-0..1 via ['fabrikantnr']
  onderhoud <-> fietsverkoop: 1-op-0..1 via ['fabrikantnr']
  fiets_inkoop <-> fietsverkoop: 1-op-1 via ['fabrikantnr']

Tabel: Fiets
  onderhoud <-> fiets_inkoop: 1-op-0..1 via ['fietsnr']
  onderhoud <-> fietsverkoop: 1-op-0..1 via ['fietsnr']
  fiets_inkoop <-> fietsverkoop: 1-op-1 via ['fietsnr']

Tabel: Filiaal
  accessoireverkoop <-> onderhoud: 1-op-0..1 via ['filiaalnr']
  accessoireverkoop <-> fietsverkoop: 1-op-1 via ['filiaalnr']
  onderhoud <-> fietsverkoop: 1-op-0..1 via ['filiaalnr']

Tabel: Klant
  accessoireverkoop <-> fietsverkoop: 1-op-0..1 via ['klantnr']

Tabel: Leverancier
  accessoire_inkoop <-> accessoireverkoop: 1-op-1 via ['leveranciernr']

Tabel: Monteur
  accessoireverkoop <-> onderhoud: 1-op-0.

In [15]:
# full refresh
logging.warning("Full Refresh-strategie geactiveerd: Foreign Key-beperkingen zijn tijdelijk uitgeschakeld.")
sdm.execute('PRAGMA foreign_keys = OFF')

# Loop door alle mappings heen
for _, _, target in MAPS:
    # Verwijder alle gegevens uit de doeltabel
    sdm.execute(f'DELETE FROM {q(target)}')

sdm.commit()

sdm.execute('PRAGMA foreign_keys = ON')
logging.info("Alle SDM-doeltabellen succesvol gewist (DELETE FROM). Foreign Key-beperkingen zijn weer geactiveerd.")

2026-04-22 14:01:47 | WARNING  | __main__:<module>:2 - Full Refresh-strategie geactiveerd: Foreign Key-beperkingen zijn tijdelijk uitgeschakeld.
2026-04-22 14:01:47 | INFO     | __main__:<module>:13 - Alle SDM-doeltabellen succesvol gewist (DELETE FROM). Foreign Key-beperkingen zijn weer geactiveerd.


In [16]:
# String-built SQL
logging.info("Gegevensoverdracht van brondatabases naar het SDM met de 'String-built SQL'-strategie begint.")

# Loop door alle mappings
for db, source_table, target_table in MAPS:
    # Haal alle rijen op uit de brontabel
    rows = sources[db].execute(
        f'SELECT * FROM {q(source_table)}'
    ).fetchall()

    # Ga verder als er geen data is
    if not rows:
        continue

    # Maak placeholders voor de INSERT-query
    placeholders = ', '.join(['?'] * len(rows[0]))

    # Voeg alle rijen toe aan de doeltabel
    sdm.executemany(
        f'INSERT INTO {q(target_table)} VALUES ({placeholders})',
        rows
    )
    
    # We gebruiken DEBUG voor gedetailleerde regel-per-regel info
    logging.debug(f"{db}.{source_table} -> {target_table} overdracht voltooid.")

sdm.commit()
logging.info("Gegevensoverdracht (Load) voltooid en wijzigingen zijn vastgelegd (commit).")

2026-04-22 14:01:47 | INFO     | __main__:<module>:2 - Gegevensoverdracht van brondatabases naar het SDM met de 'String-built SQL'-strategie begint.
2026-04-22 14:01:47 | INFO     | __main__:<module>:28 - Gegevensoverdracht (Load) voltooid en wijzigingen zijn vastgelegd (commit).


In [17]:
logging.info("Validatie van het aantal rijen van de overgedragen gegevens wordt gestart.")

# Haal alle gewone tabellen op
tables = [
    r[0] for r in sdm.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    )
]

# Loop door alle tabellen
for t in tables:
    # Tel het aantal rijen in de tabel
    count = sdm.execute(f'SELECT COUNT(*) FROM {q(t)}').fetchone()[0]

    # Toon tabelnaam en aantal rijen als DEBUG
    logging.debug(f"Validatieresultaat: {t} bevatte {count} rijen.")

2026-04-22 14:01:47 | INFO     | __main__:<module>:1 - Validatie van het aantal rijen van de overgedragen gegevens wordt gestart.


In [18]:
logging.info("Alle verbindingen met bron- en doeldatabases worden gesloten.")

for c in sources.values():
    c.close()

sdm.close()

logging.info("Het SDM-datalaadproces is succesvol en zonder fouten voltooid.")

2026-04-22 14:01:47 | INFO     | __main__:<module>:1 - Alle verbindingen met bron- en doeldatabases worden gesloten.
2026-04-22 14:01:47 | INFO     | __main__:<module>:8 - Het SDM-datalaadproces is succesvol en zonder fouten voltooid.
